# netflix-movies-and-tv-shows-dataset-ETL,EDA,Visual


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Constants
DATA_DIR = "/kaggle/input/netflix-movies-and-tv-shows-dataset/netflix_titles.csv"

# 1. Load Dataset
try:
    df = pd.read_csv(DATA_DIR)
except Exception as e:
    # Fallback for local testing if path differs, though the prompt requires the exact path
    print(f"Error loading file from {DATA_DIR}: {e}")
    # In a real Kaggle environment, this path is expected to be correct.

# 2. Basic Inspection
print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())
print("\nColumn Information:")
df.info()

# 3. ETL - Data Cleaning and Preparation
# Fill missing categorical values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('UR')

# Safe Date Conversion
# Removing leading/trailing spaces from date_added
df['date_added'] = df['date_added'].str.strip()
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract Year and Month from date_added
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# 4. Exploratory Data Analysis (EDA)
print("\nSummary Statistics for Numeric Columns:")
print(df.describe())

print("\nContent Type Distribution:")
print(df['type'].value_counts())

# 5. Visualization
sns.set_theme(style="whitegrid")

# Create a figure with multiple subplots
plt.figure(figsize=(20, 15))

# Plot 1: Distribution of Content Types
plt.subplot(2, 2, 1)
sns.countplot(data=df, x='type', palette='viridis')
plt.title('Distribution of Movies vs TV Shows')
plt.xlabel('Type')
plt.ylabel('Count')

# Plot 2: Content Release Year Distribution (Last 20 years)
plt.subplot(2, 2, 2)
recent_content = df[df['release_year'] > 2000]
sns.histplot(data=recent_content, x='release_year', hue='type', multiple='stack', bins=20, kde=True)
plt.title('Content Release Trends (Post-2000)')
plt.xlabel('Release Year')
plt.ylabel('Frequency')

# Plot 3: Top 10 Countries producing content
plt.subplot(2, 2, 3)
top_countries = df['country'].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, palette='magma')
plt.title('Top 10 Countries by Content Production')
plt.xlabel('Count')
plt.ylabel('Country')

# Plot 4: Content Added Over Years
plt.subplot(2, 2, 4)
content_by_year = df.groupby(['year_added', 'type']).size().reset_index(name='count')
sns.lineplot(data=content_by_year, x='year_added', y='count', hue='type', marker='o')
plt.title('Content Added to Netflix Over Time')
plt.xlabel('Year Added')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# Top Genres Analysis (Multi-label parsing)
print("\nTop 10 Genres (listed_in):")
genres = df['listed_in'].str.split(', ').explode()
print(genres.value_counts().head(10))

# Sample of cleaned data
print("\nCleaned Dataset Sample:")
print(df[['title', 'type', 'country', 'release_year', 'rating']].sample(5))
